# 12 · Grafana local — indicador v3 y procesos Gold

**Antes:** ejecutar el notebook 09 hasta COMPLETADA. Este notebook publica resultados existentes y Gold; no ejecuta el ETL ni el modelo. Abre Docker Desktop. Las dependencias se instalan exclusivamente al preparar `environment.yml`, nunca aquí.

Los tableros v3 se crean aparte de los históricos v2. Contratos y procesos permanecen separados. Todas las filas se conservan; el detalle usa páginas SQL de 200 registros.


In [1]:
from pathlib import Path
import sys,json
from IPython.display import display,Markdown
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'secop_pipeline').is_dir()),None)
if ROOT is None:raise RuntimeError('Abra el notebook dentro del proyecto.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from secop_pipeline.indicador_v3 import latest_result
from secop_pipeline.indicador_v3_publicacion import inspect_result
from grafana_secop.runtime import config,connection_args,execute,docker_up,configure_reader,inspect_database
from grafana_secop.seleccion_gold import describe_processes
CFG=config(ROOT)
RESULTADOS=latest_result(ROOT)
_,reporte,_=inspect_result(RESULTADOS)
PUBLICAR=True
INICIAR_POSTGRES_ETL=True # False si PostgreSQL local se administra fuera de Docker.
INICIAR_GRAFANA=True
ACTUALIZAR_DISENO=True # Regenera y recarga la configuración v3 de Grafana.
display({'lote':str(RESULTADOS),'version':reporte['version'],'contratos':reporte['resumen']['contratos']})
display(describe_processes(ROOT,CFG))


{'lote': 'C:\\Users\\luna3\\OneDrive\\Desktop\\secop_pipeline_equipo\\salida\\alertas\\indicador_v3\\f8045bc1b518250ab57fc739\\resultados.parquet',
 'version': 'alertas-cd-3.0',
 'contratos': 4353720}

{'fuente': 'C:\\Users\\luna3\\OneDrive\\Desktop\\secop_pipeline_equipo\\data\\gold\\procesos_modelo',
 'archivos': 8,
 'filas': 626821,
 'bytes': 46791656,
 'fecha_min_observada_footer': '2017-01-02 00:00:00',
 'fecha_max_observada_footer': '2026-09-21 00:00:00',
 'estadisticas_fecha_completas': True,
 'corte_declarado': None,
 'nota': 'Las fechas observadas no acreditan por sí mismas el corte de extracción.'}

## Iniciar servicios locales

La celda realiza Compose: no necesitas repetir el comando en terminal. Credenciales locales en `.env` raíz y `grafana_secop/.env`. Si PostgreSQL ya existe, se conserva; no se borran volúmenes. Este dashboard no se conecta a AWS.


In [3]:
if not (ROOT/'grafana_secop/.env').exists():raise RuntimeError('Configure grafana_secop/.env a partir de .env.example.')
connection_args(ROOT,CFG)
if CFG['database_mode']=='existing' and INICIAR_POSTGRES_ETL:
    execute(['docker','compose','up','-d','--wait','--wait-timeout',str(CFG['docker_timeout']),'postgres'],ROOT,CFG['docker_timeout']+15)
if INICIAR_GRAFANA:docker_up(ROOT,CFG)
display(inspect_database(ROOT,CFG))
configure_reader(ROOT,CFG)


 Container secop_postgres Running 
 Container secop_postgres Waiting 
 Container secop_postgres Healthy 
Finalizado en 1.9 s
 Container secop-grafana-grafana-1 Starting 
 Container secop-grafana-grafana-1 Started 
 Container secop-grafana-grafana-1 Waiting 
 Container secop-grafana-grafana-1 Healthy 
Finalizado en 2.3 s


{'database': 'secop',
 'port': 5432,
 'tables': [('gold', 'contratos_control', 0),
  ('gold', 'procesos_modelo', 0),
  ('secop', 'contratos', 0),
  ('secop', 'grupos', 0),
  ('secop', 'procesos', 0),
  ('secop', 'proveedores', 0),
  ('secop_panel', 'cubo_contratos', 0),
  ('secop_panel', 'cubo_procesos', 0),
  ('secop_panel', 'publication_current', 0),
  ('secop_panel', 'publication_products', 0),
  ('secop_panel', 'src_alerts_0e21d3f29bf3441b', 0),
  ('secop_panel', 'src_alerts_51c5fc99989f4583', 0),
  ('secop_panel', 'src_processes_ece76675f9064c7d', 0),
  ('secop_v3', 'cubo_contratos', 0),
  ('secop_v3', 'cubo_procesos', 0),
  ('secop_v3', 'productos', 0),
  ('secop_v3', 'publicacion', 0),
  ('secop_v3', 'puntajes', 0)],
 'published': 'secop_panel.publication_current'}

## Publicar

Solo se transfieren productos nuevos o modificados. Se reutiliza Gold ya publicado cuando su registro de contenido es válido. La publicación v3 usa el esquema `secop_v3` y conserva las tablas históricas. El SHA-256 comprueba que el Gold contractual corresponde al lote. No mezcles resultados de versiones distintas.


In [3]:
if PUBLICAR:
    execute([sys.executable,'-u',str(ROOT/'grafana_secop/cargar_datos.py'),'--root',str(ROOT),'--resultados',str(RESULTADOS)],ROOT,CFG['publication_timeout'])
if ACTUALIZAR_DISENO:
    execute([sys.executable,str(ROOT/'grafana_secop/generar_dashboards_v3.py')],ROOT,60)
    execute(['docker','compose','restart','grafana'],ROOT/'grafana_secop',CFG['docker_timeout'])


{"etapa": "gold", "estado": "REUTILIZADO", "transcurrido_s": 3.436, "rss_mb": 128.3}
{"etapa": "transferencia_alerts", "estado": "INICIO", "transcurrido_s": 3.452, "rss_mb": 130.2}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 8.533, "rss_mb": 191.1, "filas": 131072, "total": 4353720}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 13.681, "rss_mb": 177.9, "filas": 262144, "total": 4353720}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 18.765, "rss_mb": 176.9, "filas": 401408, "total": 4353720}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 23.854, "rss_mb": 177.6, "filas": 536576, "total": 4353720}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 28.849, "rss_mb": 178.5, "filas": 671744, "total": 4353720}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 33.87, "rss_mb": 179.8, "filas": 798720, "total": 4353720}
{"etapa": "alerts", "estado": "COPY", "transcurrido_s": 38.971, "rss_mb": 180.0, "filas": 937984, "total": 4353720}
{"etapa":

## Comprobar y abrir

La regla visual v3 informa que existen contratos priorizados por cupo trimestral. No aplica la regla anterior de dos de tres. Error/NoData no equivalen a ausencia de riesgo. La interpretación y parámetros están en cada tablero. El filtro Año cambia gráficos; clasificación e ID afectan al detalle. Restaura Página=1 después de cambiar filtros.


In [4]:
import psycopg
with psycopg.connect(**connection_args(ROOT,CFG)) as conn:
    with conn.transaction():
        conn.execute('SET TRANSACTION READ ONLY')
        if conn.execute("SELECT to_regclass('secop_v3.dash_metadatos')").fetchone()[0] is None:
            print('Falta publicar v3; activa PUBLICAR=True.')
        else:
            display(conn.execute('SELECT * FROM secop_v3.dash_metadatos').fetchall())
            display(conn.execute('SELECT dimension,sum(evaluados),sum(alertas),sum(no_evaluables) FROM secop_v3.dash_dimensiones GROUP BY 1 ORDER BY 1').fetchall())
from dotenv import dotenv_values
url='http://localhost:'+dotenv_values(ROOT/'grafana_secop/.env').get('GRAFANA_PORT','3001')
display(Markdown(f'- [Contratos: indicador v3]({url}/d/secop-contratos-v3)\n- [Procesos Gold]({url}/d/secop-procesos-v3)'))


[('3a10d5f08b5a45a7ee17f973bff977811e1e0b62340108b075b9e01755fe6d85',
  'alertas-cd-3.0',
  '2026-09-21',
  '2000',
  '4353720',
  '626821',
  None,
  '2017-01-02 00:00:00',
  '2026-09-21 00:00:00',
  'GOLD_EXISTENTE_SELECCIONADO',
  True)]

[('C1', Decimal('4330105'), Decimal('206'), Decimal('23615')),
 ('C2', Decimal('4343765'), Decimal('369'), Decimal('9955')),
 ('C3', Decimal('262261'), Decimal('69082'), Decimal('4091459')),
 ('P1', Decimal('4349673'), Decimal('1024545'), Decimal('4047')),
 ('P2', Decimal('248202'), Decimal('57264'), Decimal('4105518')),
 ('V1', Decimal('4246400'), Decimal('13340'), Decimal('107320')),
 ('V2', Decimal('4332972'), Decimal('13057'), Decimal('20748'))]

- [Contratos: indicador v3](http://localhost:3001/d/secop-contratos-v3)
- [Procesos Gold](http://localhost:3001/d/secop-procesos-v3)